# Inventory-impact coordinates: L vs skew vs frozen-grid K

Three concentration charges, one objective. The [inventory-impact reference](../docs/design/inventory_impact_reference.md) measures risk-adjusted return

```text
R = F / K,    K = E[payout | worst 5%] - E[payout]
```

`F` is ordinary fee income plus inventory income. `K` is always the fine-cell economic capital of the finished book, never the coordinate a fee is computed from. This notebook asks which of the three implemented coordinates puts a **matched** inventory-income total onto the books that actually consume that capital.

| Mechanism | Source | Coordinate | Charge |
| --- | --- | --- | --- |
| **L** (mainnet occupancy) | shipped inventory impact | `L = M + λ(T − M)`, `λ = 0.25` | capped convex potential on `L` |
| **Skew** (DBU-732) | `at/dbu-732-predict-skew-charge` | payout stdev over the settlement line | `r · Δσ`, signed / path-independent |
| **K grid** (this branch) | frozen-grid prototype | average of the five worst 1% bucket maxima, minus `E` | capped convex potential on `K_grid`, no rebate |

The numerator is held constant in aggregate: after the K-grid charge at an 8% maximum marginal rate sets a target, L's maximum marginal rate and the skew rate are fitted so all three collect the same total inventory dollars across the paired books. Differences in `R` are then differences in *where* that income lands, not in how much was billed. Eight percent is a heavier comparison rate than the 2% figure used elsewhere in this series; it is not a production recommendation. At this pot the targeting gap is easier to read: K's residual spread is about half of skew's.

Flow, fee formula, and `K` match [the value-ceiling notebook](inventory_impact_value_ceiling.ipynb). Opens only; no trader relocation. Skew is evaluated over every cell — the DBU-732 window is tenor-scaled to about ±5–6 daily standard deviations, which on this equal-probability line is the whole book.

A later section classifies each contract by **where it lands** (on the current tail, on the book but off the tail, or disjoint) and by whether it **adds true capital**. The test for a good charge is that it bills tail-adding flow and leaves non-risk flow cheap.

In [1]:
"""Primitives shared with the value-ceiling notebook, plus the three coordinates."""

from __future__ import annotations

from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

CELLS = 2_000
TAIL_CELLS = CELLS // 20
GRID_BUCKETS = 100
CELLS_PER_GRID_BUCKET = CELLS // GRID_BUCKETS
GRID_TAIL_BUCKETS = GRID_BUCKETS // 20
assert TAIL_CELLS * 20 == CELLS
assert CELLS_PER_GRID_BUCKET * GRID_BUCKETS == CELLS

BASE_FEE_RATE = 0.02
FEE_RAMP = 1.0
LAMBDA = 0.25
CONVEX_MAX_RATE = 0.08

WIDTH_MASSES = np.array([0.005, 0.01, 0.02, 0.05, 0.10, 0.20])
THEME_SPREAD_CELLS = 40
MEDIAN_QUANTITY = 1_000.0
QUANTITY_LOG_SD = 0.8
TRADES_PER_MARKET = 200
COMPARE_CLUSTERING = [0.0, 0.25, 0.5, 0.75, 1.0]
COMPARE_SEEDS = 16


@dataclass(frozen=True)
class Trade:
    lower: int
    upper: int
    quantity: float

    @property
    def mass(self) -> float:
        return (self.upper - self.lower) / CELLS

    @property
    def fee(self) -> float:
        mass = self.mass
        return BASE_FEE_RATE * float(np.sqrt(mass * (1.0 - mass))) * FEE_RAMP * self.quantity


def payout_profile(trades) -> np.ndarray:
    edges = np.zeros(CELLS + 1)
    for trade in trades:
        edges[trade.lower] += trade.quantity
        edges[trade.upper] -= trade.quantity
    return np.cumsum(edges[:-1])


def economic_capital(payout: np.ndarray) -> float:
    cut = CELLS - TAIL_CELLS
    tail = np.partition(payout, cut)[cut:]
    return float(tail.mean() - payout.mean())


def grid_capital(payout: np.ndarray) -> float:
    maxima = payout.reshape(GRID_BUCKETS, CELLS_PER_GRID_BUCKET).max(axis=1)
    cut = GRID_BUCKETS - GRID_TAIL_BUCKETS
    tail = np.partition(maxima, cut)[cut:]
    return float(tail.mean() - payout.mean())


def payout_liability(payout: np.ndarray, total_max: float) -> float:
    peak = float(payout.max()) if payout.size else 0.0
    gap = max(0.0, total_max - peak)
    return peak + LAMBDA * gap


def skew_sigma(payout: np.ndarray) -> float:
    """Tick-weighted population stdev of the payout profile, the DBU-732 statistic."""
    return float(payout.std(ddof=0))


def convex_units(capital: np.ndarray, scale: float) -> np.ndarray:
    """Phi[r, B] / r, so income is linear in the fitted rate."""
    capital = np.maximum(np.asarray(capital, dtype=float), 0.0)
    if scale <= 0.0:
        return CONVEX_MAX_RATE * capital / CONVEX_MAX_RATE
    below = capital * capital / (2.0 * scale)
    above = scale / 2.0 + (capital - scale)
    return np.where(capital <= scale, below, above)


def convex_path_charges(path: np.ndarray, scale: float, rate: float) -> np.ndarray:
    units = convex_units(path, scale)
    return rate * np.maximum(0.0, np.diff(units))


_parity = [
    ("guaranteed full-line payout", [Trade(0, CELLS, 1_000.0)], 0.0),
    ("five-bucket pile-on, 5% mass", [Trade(0, CELLS // 20, 1_000.0)], 950.0),
    ("ten-bucket plateau, 10% mass", [Trade(0, CELLS // 10, 900.0)], 810.0),
]
for name, book, expected in _parity:
    payout = payout_profile(book)
    assert abs(economic_capital(payout) - expected) < 1e-9, name

full_line = payout_profile([Trade(0, CELLS, 1_000.0)])
assert abs(skew_sigma(full_line)) < 1e-9
assert abs(payout_liability(full_line, 1_000.0) - 1_000.0) < 1e-9

print(f"{CELLS} cells, {GRID_BUCKETS} grid buckets, λ={LAMBDA}")
print("K parity and the full-line σ=0 / L=M checks passed.")

2000 cells, 100 grid buckets, λ=0.25
K parity and the full-line σ=0 / L=M checks passed.


## Method

Paired books: within each seed, clustering only switches which placement each trade uses. Widths, quantities, and both placement draws are shared. 200 trades, 16 seeds, clustering `{0, 0.25, 0.5, 0.75, 1}`.

The K-grid charge at `r_max = 8%` and `B =` median final `K_grid` at 50% clustering sets the inventory-income target. L uses the same convex shape on `L` with `B =` median final `L` at 50% clustering; its `r_max` is fitted. Skew is linear in terminal `σ`; its rate is fitted. Opens never lower `L` or (under no-rebate K) pay a refund, so those two nets equal the potential at the terminal state. Skew is signed, so its net on a book is `r · σ_final`. This 8% pot is about four times the inventory dollars of the 2% run; L and skew are rematched to that larger pot, not left at their 2% rates.

```text
R(mechanism, clustering) = mean_seeds (ordinary fees + inventory income) / K_true
spread                    = R(dispersed) / R(clustered)
```

In [2]:
"""Walk paired books, match inventory income, compare R."""


def paired_flows(n_trades: int, seed: int) -> dict[float, list[Trade]]:
    rng = np.random.default_rng(seed)
    theme = int(rng.integers(0, CELLS))
    widths = np.rint(rng.choice(WIDTH_MASSES, size=n_trades) * CELLS).astype(int)
    quantities = MEDIAN_QUANTITY * np.exp(rng.normal(0.0, QUANTITY_LOG_SD, size=n_trades))
    clustered_draw = np.rint(rng.normal(theme, THEME_SPREAD_CELLS, size=n_trades)).astype(int)
    independent_draw = rng.integers(0, CELLS, size=n_trades)
    cluster_draw = rng.random(n_trades)
    flows = {}
    for clustering in COMPARE_CLUSTERING:
        centers = np.where(cluster_draw < clustering, clustered_draw, independent_draw)
        trades = []
        for width, quantity, center in zip(widths, quantities, centers, strict=True):
            lower = int(np.clip(center - width // 2, 0, CELLS - width))
            trades.append(Trade(lower, lower + int(width), float(quantity)))
        flows[clustering] = trades
    return flows


def charge_trace(trades: list[Trade]) -> dict:
    payout = np.zeros(CELLS)
    total_max = 0.0
    k_path = [0.0]
    l_path = [0.0]
    sigma_path = [0.0]
    fees = 0.0
    for trade in trades:
        payout[trade.lower:trade.upper] += trade.quantity
        total_max += trade.quantity
        k_path.append(grid_capital(payout))
        l_path.append(payout_liability(payout, total_max))
        sigma_path.append(skew_sigma(payout))
        fees += trade.fee
    return {
        "k path": np.asarray(k_path),
        "l path": np.asarray(l_path),
        "sigma path": np.asarray(sigma_path),
        "ordinary fees": fees,
        "true K": economic_capital(payout),
        "grid K": float(k_path[-1]),
        "L": float(l_path[-1]),
        "sigma": float(sigma_path[-1]),
    }


traces: dict[tuple[float, int], dict] = {}
for seed in range(COMPARE_SEEDS):
    for clustering, flow in paired_flows(TRADES_PER_MARKET, 80_000 + seed).items():
        traces[(clustering, seed)] = charge_trace(flow)

keys = [(clustering, seed) for clustering in COMPARE_CLUSTERING for seed in range(COMPARE_SEEDS)]
b_k = float(np.median([traces[(0.5, seed)]["grid K"] for seed in range(COMPARE_SEEDS)]))
b_l = float(np.median([traces[(0.5, seed)]["L"] for seed in range(COMPARE_SEEDS)]))

k_income = {
    key: float(convex_path_charges(traces[key]["k path"], b_k, CONVEX_MAX_RATE).sum())
    for key in keys
}
target = float(sum(k_income.values()))
l_units = {key: float(convex_units(np.array([traces[key]["L"]]), b_l)[0]) for key in keys}
sigma_final = {key: traces[key]["sigma"] for key in keys}
r_l = target / float(sum(l_units.values()))
r_skew = target / float(sum(sigma_final.values()))
l_income = {key: r_l * units for key, units in l_units.items()}
skew_income = {key: r_skew * sigma_final[key] for key in keys}

mechanisms = {
    "baseline": {key: 0.0 for key in keys},
    "L": l_income,
    "skew": skew_income,
    "K grid": k_income,
}

shape_rows = []
seed_shape_r = {name: {seed: [] for seed in range(COMPARE_SEEDS)} for name in mechanisms}
for clustering in COMPARE_CLUSTERING:
    for seed in range(COMPARE_SEEDS):
        key = (clustering, seed)
        trace = traces[key]
        for name, incomes in mechanisms.items():
            r = (trace["ordinary fees"] + incomes[key]) / trace["true K"]
            seed_shape_r[name][seed].append(r)
            shape_rows.append({
                "clustering": clustering,
                "seed": seed,
                "mechanism": name,
                "R": r,
                "inventory income": incomes[key],
                "ordinary fees": trace["ordinary fees"],
                "K": trace["true K"],
            })

shape = pd.DataFrame(shape_rows)
summary = (
    shape.groupby(["mechanism", "clustering"], sort=False)["R"]
    .mean()
    .unstack("clustering")
)
summary["spread"] = summary[0.0] / summary[1.0]
order = ["baseline", "L", "skew", "K grid"]
summary = summary.loc[order]

paired = []
for seed in range(COMPARE_SEEDS):
    row = {"seed": seed}
    for name in order:
        values = seed_shape_r[name][seed]
        row[f"{name} spread"] = max(values) / min(values)
    paired.append(row)
paired = pd.DataFrame(paired)

ordinary_total = sum(traces[key]["ordinary fees"] for key in keys)
print(f"books: {len(traces)} | B_K=${b_k:,.0f} | B_L=${b_l:,.0f}")
print(f"matched inventory income ${target:,.2f}  ({target / ordinary_total:.1%} of ordinary fees)")
print(f"fitted r_L={r_l:.4%}   r_skew={r_skew:.4%}   r_K={CONVEX_MAX_RATE:.2%} (fixed)")
print()
print(
    "spread reduction vs L:  "
    f"skew {1.0 - summary.loc['skew', 'spread'] / summary.loc['L', 'spread']:.1%}   "
    f"K grid {1.0 - summary.loc['K grid', 'spread'] / summary.loc['L', 'spread']:.1%}"
)
print(
    "paired spread vs L (mean ± SE):  "
    f"skew {1.0 - (paired['skew spread'] / paired['L spread']).mean():.1%} "
    f"± {(paired['skew spread'] / paired['L spread']).std(ddof=1) / np.sqrt(COMPARE_SEEDS):.1%}   "
    f"K grid {1.0 - (paired['K grid spread'] / paired['L spread']).mean():.1%} "
    f"± {(paired['K grid spread'] / paired['L spread']).std(ddof=1) / np.sqrt(COMPARE_SEEDS):.1%}"
)

display(summary.style.format("{:.4f}").format({"spread": "{:.2f}x"}).set_caption(
    "Risk-adjusted return by flow shape, matched inventory income"
))

books: 80 | B_K=$66,889 | B_L=$138,166
matched inventory income $292,596.13  (333.8% of ordinary fees)
fitted r_L=4.9118%   r_skew=18.1599%   r_K=8.00% (fixed)

spread reduction vs L:  skew 26.1%   K grid 60.6%
paired spread vs L (mean ± SE):  skew 26.0% ± 2.1%   K grid 55.9% ± 0.7%


clustering,0.000000,0.250000,0.500000,0.750000,1.000000,spread
mechanism,,,,,,
baseline,0.088036,0.037594,0.017690,0.011540,0.008659,10.17x
L,0.213065,0.112035,0.072919,0.062180,0.056832,3.75x
skew,0.173359,0.097775,0.071787,0.065639,0.062578,2.77x
K grid,0.107776,0.067512,0.066532,0.071705,0.072885,1.48x


In [3]:
"""Return-per-dollar-of-risk figure, same axes as the reference doc."""

means = shape.groupby(["mechanism", "clustering"])["R"].mean().unstack("mechanism")[order]
errors = shape.groupby(["mechanism", "clustering"])["R"].std().unstack("mechanism")[order]
errors = errors / np.sqrt(COMPARE_SEEDS)

figure, axis = plt.subplots(figsize=(7.2, 4.2))
colors = {
    "baseline": "#6b7280",
    "L": "#b45309",
    "skew": "#7c3aed",
    "K grid": "#0f766e",
}
labels = {
    "baseline": "ordinary fee only",
    "L": "L (mainnet occupancy)",
    "skew": "skew (DBU-732)",
    "K grid": "K grid (frozen-grid)",
}
for name in order:
    axis.errorbar(
        means.index,
        means[name],
        yerr=errors[name],
        marker="o",
        color=colors[name],
        label=labels[name],
    )
axis.set(
    xlabel="flow clustering",
    ylabel="risk-adjusted return  F / K",
    title="Same inventory dollars, different targeting",
)
axis.legend(frameon=False)
axis.set_ylim(bottom=0)
figure.tight_layout()
plt.show()

clustered = shape[shape["clustering"] == 1.0].groupby("mechanism")["R"].mean().loc[order]
print("fully clustered R")
for name in order:
    print(f"  {labels[name]:<28} {clustered[name]:.4f}   vs L: {clustered[name] / clustered['L'] - 1.0:+.1%}")

fully clustered R
  ordinary fee only            0.0087   vs L: -84.8%
  L (mainnet occupancy)        0.0568   vs L: +0.0%
  skew (DBU-732)               0.0626   vs L: +10.1%
  K grid (frozen-grid)         0.0729   vs L: +28.2%


/var/folders/ht/2hvnpbts133dyljvs6p8y_cr0000gn/T/ipykernel_73352/2127763349.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Per-contract fees by placement

The book-level spread says which mechanism raises `R` on piled markets. It does not say whether a *safe* order still gets billed.

Each open is classified two ways, using the book as it stood when the order arrived:

- **Risk:** did true `K` rise, fall, or stay put?
- **Where:** did the range sit on the current worst 5% of cells (`on tail`), overlap the book but miss that tail (`on book, off tail`), miss the book entirely (`disjoint`), or open an empty book?

Fees use the same matched rates as the `R` comparison. The figure is the readout: three placements, three charges. A charge that tracks the tail is ~$0 on disjoint and off-tail flow and large on the current tail.

In [4]:
"""Per-trade fees by true capital change and by where the range sat."""


def classify_placement(payout: np.ndarray, trade: Trade) -> str:
    if float(payout.max()) <= 0.0:
        return "opens book"
    span = payout[trade.lower:trade.upper]
    tail_floor = float(np.partition(payout, CELLS - TAIL_CELLS)[CELLS - TAIL_CELLS])
    if tail_floor > 0.0 and np.any(span >= tail_floor):
        return "on tail"
    if np.any(span > 0.0):
        return "on book, off tail"
    return "disjoint"


def classify_risk(delta_k: float) -> str:
    if delta_k > 1e-9:
        return "adds risk"
    if delta_k < -1e-9:
        return "reduces risk"
    return "no change"


rows = []
for clustering in COMPARE_CLUSTERING:
    for seed in range(COMPARE_SEEDS):
        trades = paired_flows(TRADES_PER_MARKET, 80_000 + seed)[clustering]
        payout = np.zeros(CELLS)
        total_max = 0.0
        true_k = 0.0
        grid_k = 0.0
        liability = 0.0
        sigma = 0.0
        for trade in trades:
            where = classify_placement(payout, trade)
            payout[trade.lower:trade.upper] += trade.quantity
            total_max += trade.quantity
            true_after = economic_capital(payout)
            grid_after = grid_capital(payout)
            l_after = payout_liability(payout, total_max)
            sigma_after = skew_sigma(payout)
            k_fee = float(convex_path_charges(np.array([grid_k, grid_after]), b_k, CONVEX_MAX_RATE)[0])
            l_fee = float(r_l * max(0.0, convex_units(np.array([l_after]), b_l)[0] - convex_units(np.array([liability]), b_l)[0]))
            skew_fee = float(r_skew * (sigma_after - sigma))
            rows.append({
                "clustering": clustering,
                "placement": where,
                "risk": classify_risk(true_after - true_k),
                "delta K": true_after - true_k,
                "ordinary fee": trade.fee,
                "L": l_fee,
                "skew": skew_fee,
                "K grid": k_fee,
            })
            true_k, grid_k, liability, sigma = true_after, grid_after, l_after, sigma_after

trades_df = pd.DataFrame(rows)
MECHS = ["L", "skew", "K grid"]


def spearman(a: pd.Series, b: pd.Series) -> float:
    return float(np.corrcoef(a.rank(), b.rank())[0, 1])


n = len(trades_df)
adds = trades_df["delta K"] > 1e-9
safe = ~adds
print(f"{n:,} opens  |  {adds.mean():.1%} add true K  |  {safe.mean():.1%} do not")
print()
print("Fee taken from flow that does not add risk  (want this small)")
leak_rows = []
for mech in MECHS:
    charged = trades_df[mech].clip(lower=0.0)
    leak = float(charged[safe].sum() / charged.sum()) if charged.sum() else 0.0
    rebate_to_safe = float((-trades_df.loc[safe, mech].clip(upper=0.0)).sum())
    leak_rows.append({
        "mechanism": mech,
        "inventory $ from non-risk opens": charged[safe].sum(),
        "share of positive inventory fees": leak,
        "rebate to non-risk opens": rebate_to_safe,
        "rank corr with ΔK": spearman(trades_df[mech], trades_df["delta K"]),
    })
leak = pd.DataFrame(leak_rows)
display(leak.style.format({
    "inventory $ from non-risk opens": "${:,.0f}",
    "share of positive inventory fees": "{:.1%}",
    "rebate to non-risk opens": "${:,.0f}",
    "rank corr with ΔK": "{:+.3f}",
}).hide(axis="index").set_caption("Non-risk leakage at matched income"))

place = []
for placement, part in trades_df.groupby("placement", sort=False):
    row = {"placement": placement, "opens": len(part), "share of opens": len(part) / n, "mean ΔK": part["delta K"].mean()}
    for mech in MECHS:
        row[f"{mech} mean fee"] = part[mech].mean()
        row[f"{mech} $ share"] = part[mech].clip(lower=0.0).sum() / trades_df[mech].clip(lower=0.0).sum()
    place.append(row)
place = pd.DataFrame(place)
display(place.style.format({
    "opens": "{:,}",
    "share of opens": "{:.1%}",
    "mean ΔK": "${:,.0f}",
    **{f"{m} mean fee": "${:,.2f}" for m in MECHS},
    **{f"{m} $ share": "{:.1%}" for m in MECHS},
}).hide(axis="index").set_caption("Where the order sat vs mean inventory fee"))

risk = []
for label, part in trades_df.groupby("risk", sort=False):
    row = {"risk": label, "opens": len(part), "share of opens": len(part) / n, "mean ΔK": part["delta K"].mean()}
    for mech in MECHS:
        row[f"{mech} mean fee"] = part[mech].mean()
    risk.append(row)
risk = pd.DataFrame(risk)
display(risk.style.format({
    "opens": "{:,}",
    "share of opens": "{:.1%}",
    "mean ΔK": "${:,.0f}",
    **{f"{m} mean fee": "${:,.2f}" for m in MECHS},
}).hide(axis="index").set_caption("True capital change vs mean inventory fee"))

adding = trades_df[adds].copy()
adding["quintile"] = pd.qcut(adding["delta K"], 5, labels=False)
rates = adding.groupby("quintile", sort=True).apply(
    lambda part: pd.Series({
        "mean ΔK": part["delta K"].mean(),
        **{mech: part[mech].sum() / part["delta K"].sum() for mech in MECHS},
    }),
    include_groups=False,
)
print()
print("Effective inventory rate on capital-adding opens, lowest→highest ΔK fifth")
for mech in MECHS:
    print(f"  {mech:<8} {rates.iloc[0][mech]:.2%} → {rates.iloc[-1][mech]:.2%}")

place_order = ["disjoint", "on book, off tail", "on tail"]
place_copy = {
    "disjoint": ("Disjoint", "new coverage, away from the book", [("skew", "taxing diversification", (0, 16))]),
    "on book, off tail": (
        "On the book, off the tail",
        "overlap, but not the worst 5%",
        [("L", "still charging overlap", (0, 16)), ("skew", "rebate — variance fell", (12, -32))],
    ),
    "on tail": ("On the tail", "this is the risk", [("K grid", "K charges only on the tail", (0, 16))]),
}
plot_names = [("L", "L"), ("skew", "Skew"), ("K grid", "K grid")]
colors = {"L": "#b0673b", "skew": "#6d28d9", "K grid": "#0f766e"}
means = trades_df.groupby("placement")[MECHS].mean()

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})
figure, axes = plt.subplots(1, 3, figsize=(11.4, 4.6), gridspec_kw={"wspace": 0.34})
x = np.arange(len(plot_names))
for axis, placement in zip(axes, place_order):
    values = [float(means.loc[placement, key]) for key, _ in plot_names]
    axis.bar(x, values, color=[colors[key] for key, _ in plot_names], width=0.72, zorder=2)
    axis.axhline(0.0, color="#9ca3af", linewidth=0.9, zorder=1)
    axis.set_xticks(x, [label for _, label in plot_names])
    low, high = min(values), max(values)
    pad = max(abs(high), 1.0) * 0.28
    axis.set_ylim(min(-1.0, low - 1.6) if low < 0 else -1.0, high + pad)
    title, subtitle, notes = place_copy[placement]
    axis.set_title(title, fontsize=11, pad=14)
    axis.text(0.5, 1.02, subtitle, transform=axis.transAxes, ha="center", va="bottom", fontsize=8, color="#4b5563")
    if axis is axes[0]:
        axis.set_ylabel("mean inventory fee ($)")
    for idx, value in enumerate(values):
        axis.annotate(
            f"${value:,.2f}",
            xy=(idx, value),
            xytext=(0, 6 if value >= 0 else -11),
            textcoords="offset points",
            ha="center",
            fontsize=8,
            color="#111827",
        )
    for mech, text, offset in notes:
        idx = [key for key, _ in plot_names].index(mech)
        value = float(means.loc[placement, mech])
        kwargs = dict(
            xy=(idx, value),
            xytext=offset,
            textcoords="offset points",
            ha="center" if offset[0] == 0 else "left",
            fontsize=8,
            color=colors[mech],
        )
        if value < 0:
            kwargs["arrowprops"] = dict(arrowstyle="->", color=colors[mech], lw=0.8)
        axis.annotate(text, **kwargs)

figure.suptitle("Where the inventory fee lands", fontsize=13, y=1.08)
figure.text(
    0.5,
    -0.06,
    "Mean inventory fee per open, same inventory-income total across L, skew, and K. "
    "16,000 opens. First-trade “opens book” (0.5% of flow) omitted.",
    ha="center",
    fontsize=8,
    color="#555555",
    style="italic",
)
plt.show()


16,000 opens  |  55.8% add true K  |  44.2% do not

Fee taken from flow that does not add risk  (want this small)


mechanism,inventory $ from non-risk opens,share of positive inventory fees,rebate to non-risk opens,rank corr with ΔK
L,"$54,041",18.5%,$0,+0.448
skew,"$14,133",4.5%,"$22,264",+0.858
K grid,$45,0.0%,$0,+0.882


placement,opens,share of opens,mean ΔK,L mean fee,L $ share,skew mean fee,skew $ share,K grid mean fee,K grid $ share
opens book,80,0.5%,$752,$0.52,0.0%,$59.88,1.5%,$0.74,0.0%
disjoint,571,3.6%,$83,$2.11,0.4%,$5.38,1.2%,$0.25,0.0%
"on book, off tail","6,725",42.0%,$-51,$8.17,18.8%,$-0.35,6.0%,$0.09,0.2%
on tail,"8,624",53.9%,$637,$27.42,80.8%,$33.29,91.3%,$33.83,99.7%


risk,opens,share of opens,mean ΔK,L mean fee,skew mean fee,K grid mean fee
adds risk,"8,927",55.8%,$641,$26.72,$33.69,$32.77
reduces risk,"7,046",44.0%,$-66,$7.66,$-1.18,$0.00
no change,27,0.2%,$-0,$3.04,$7.12,$0.44



Effective inventory rate on capital-adding opens, lowest→highest ΔK fifth
  L        10.41% → 3.65%
  skew     6.62% → 5.37%
  K grid   4.82% → 5.18%
wrote docs/assets/inventory-impact-placement-cost.png


/var/folders/ht/2hvnpbts133dyljvs6p8y_cr0000gn/T/ipykernel_73352/1963854211.py:210: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Reading

Baseline `R` falls as flow concentrates because ordinary fees barely move while true `K` rises. A useful inventory charge raises clustered `R` relative to dispersed `R` and therefore shrinks the spread, without collecting more in total than the other two.

L fails translation invariance: a fully funded book still scores `L = M`. It also cannot go negative, so a disjoint open is billed for `λ · quantity` even when peak payout is unchanged. Skew is translation-invariant — a complete cover has `σ = 0` — but it is a variance of the whole profile, so it charges a book that is merely uneven in a harmless place the same as one whose unevenness sits in the tail.

The frozen-grid coordinate is the one whose level is the same quantity as the denominator, coarsened. At matched income it should shrink the spread the most if the 100-bucket estimator ranks books the way true `K` does.

The placement figure asks a sharper question: of those matched dollars, how many were taken from opens that did **not** sit on the tail? L still bills overlap. Skew rebates off-tail overlap and taxes disjoint coverage. K is ~$0 except on the tail.

Limits: generated flow, opens only, no staleness, real-valued arithmetic, skew window taken as the full line. Matching aggregate income does not equalize per-trade bills. The 8% K rate is a comparison choice so the residual-spread gap is easy to see; it is not a calibrated production rate, and the same pot makes L and skew look better than they do at 2%.